# Plot contours {#ref_tutorials_plot_contour}

This tutorial shows different commands for plotting data contours on
meshes.

PyDPF-Core has a variety of plotting methods for generating 3D plots
with Python. These methods use VTK and leverage the
[PyVista](https://github.com/pyvista/pyvista) library.


# Load data to plot

## Load a result file in a model


In [ ]:
import ansys.dpf.core as dpf
from ansys.dpf.core import examples, operators as ops

result_file_path_1 = examples.download_piston_rod()
model_1 = dpf.Model(data_sources=result_file_path_1)

# Extract data for the contour

For more information about extracting results from a result file, see
the `ref_tutorials_import_data`{.interpreted-text role="ref"} tutorials
section.

:::: note
::: title
Note
:::

Only the `elemental` or `nodal` locations are supported for plotting.
::::

Here, we choose to plot the XX component of the stress tensor.


In [ ]:
stress_XX_op = ops.result.stress_X(data_sources=model_1)

# The default behavior is to return data as ElementalNodal
print(stress_XX_op.eval())

Request the stress in a `nodal` location (the default `ElementalNodal`
location is not supported for plotting). We define the new location
using the operator input. Another option would be using the
`to_nodal_fc<ansys.dpf.core.operators.averaging.to_nodal_fc.to_nodal_fc>`{.interpreted-text
role="class"} averaging operator on the output of the stress operator.


In [ ]:
stress_XX_op.inputs.requested_location(dpf.locations.nodal)
stress_XX_fc = stress_XX_op.eval()

# Extract the mesh


In [ ]:
meshed_region_1 = model_1.metadata.meshed_region

# Plot a contour of a single field

There are three methods to plot a single
`Field<ansys.dpf.core.field.Field>`{.interpreted-text role="class"}:

- `Field.plot()<ansys.dpf.core.field.Field.plot>`{.interpreted-text
  role="py:meth"}
- `MeshedRegion.plot()<ansys.dpf.core.meshed_region.MeshedRegion.plot>`{.interpreted-text
  role="py:meth"} with the field as argument
- `DpfPlotter<ansys.dpf.core.plotter.DpfPlotter>`{.interpreted-text
  role="class"} with
  `add_field()<ansys.dpf.core.plotter.DpfPlotter.add_field>`{.interpreted-text
  role="py:meth"} (more performant)

Get a single field from the `FieldsContainer`.


In [ ]:
stress_XX = stress_XX_fc[0]

# Plot using `Field.plot()`

If the `Field<ansys.dpf.core.field.Field>`{.interpreted-text
role="class"} does not have an associated mesh support (see
`Field.meshed_region<ansys.dpf.core.field.Field.meshed_region>`{.interpreted-text
role="py:attr"}), provide a mesh with the `meshed_region` argument.


In [ ]:
stress_XX.plot(meshed_region=meshed_region_1)

# Plot using `MeshedRegion.plot()`

Use the `field_or_fields_container` argument to pass the field.


In [ ]:
meshed_region_1.plot(field_or_fields_container=stress_XX)

# Plot using `DpfPlotter`

1.  Create an instance of
    `DpfPlotter<ansys.dpf.core.plotter.DpfPlotter>`{.interpreted-text
    role="class"}.
2.  Add the field using
    `add_field()<ansys.dpf.core.plotter.DpfPlotter.add_field>`{.interpreted-text
    role="py:meth"}. If the field has no associated mesh support,
    provide a mesh with the `meshed_region` argument.
3.  Render and show the figure using
    `show_figure()<ansys.dpf.core.plotter.DpfPlotter.show_figure>`{.interpreted-text
    role="py:meth"}.

You can also first call
`add_mesh()<ansys.dpf.core.plotter.DpfPlotter.add_mesh>`{.interpreted-text
role="py:meth"} to add the mesh and then call `add_field()` without the
`meshed_region` argument.


In [ ]:
plotter_1 = dpf.plotter.DpfPlotter()
plotter_1.add_field(field=stress_XX, meshed_region=meshed_region_1)
plotter_1.show_figure()

# Plot a contour of multiple fields

## Prepare a collection of fields

:::: warning
::: title
Warning
:::

The fields should not have conflicting data --- you cannot build a
contour for two fields with two different sets of data for the same mesh
entities (intersecting scopings). These methods are therefore not
available for a collection of fields varying across time, or for
different shell layers of the same elements.
::::

Here we split the field for XX stress based on material to get a
collection of fields with non-conflicting associated mesh entities.

We use the
`split_fields<ansys.dpf.core.operators.mesh.split_fields.split_fields>`{.interpreted-text
role="class"} operator together with the
`split_mesh<ansys.dpf.core.operators.mesh.split_mesh.split_mesh>`{.interpreted-text
role="class"} operator. For MAPDL results, a split on material is
equivalent to a split on bodies.


In [ ]:
fields = (
    ops.mesh.split_fields(
        field_or_fields_container=stress_XX_fc,
        meshes=ops.mesh.split_mesh(mesh=meshed_region_1, property="mat"),
    )
).eval()
print(fields)

# Plot the contour using `FieldsContainer.plot()`

Use
`FieldsContainer.plot()<ansys.dpf.core.fields_container.FieldsContainer.plot>`{.interpreted-text
role="py:meth"}.


In [ ]:
fields.plot()

The `label_space` argument provides further field filtering
capabilities.


In [ ]:
fields.plot(label_space={"mat": 1})

# Plot the contour using `MeshedRegion.plot()`

Use the `field_or_fields_container` argument.


In [ ]:
meshed_region_1.plot(field_or_fields_container=fields)

# Plot the contour using `DpfPlotter`

Add each field individually using
`add_field()<ansys.dpf.core.plotter.DpfPlotter.add_field>`{.interpreted-text
role="py:meth"}.


In [ ]:
plotter_2 = dpf.plotter.DpfPlotter()
plotter_2.add_field(field=fields[0])
plotter_2.add_field(field=fields[1])
plotter_2.show_figure()